# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


# **Load Data**

In [5]:
# Load datasets
folds = paths.load_cv_folds(k=5)

In [6]:
def evaluate_recommender(recommender, at, URM_validation):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Hyperparameter search**

In [7]:
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender

optimizer = ModelOptimizer("UserKNN_tversky")

STUDY_NAME = UserKNNCFRecommender.RECOMMENDER_NAME + '_tversky'

In [8]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "tversky",
        "topK": optuna_trial.suggest_int("topK", 5, 500),
        "shrink": optuna_trial.suggest_int("shrink", 0, 1000),
        "tversky_alpha": optuna_trial.suggest_float("tversky_alpha", 0.0, 1.0),
        "tversky_beta": optuna_trial.suggest_float("tversky_beta", 0.0, 1.0),
        "normalize": optuna_trial.suggest_categorical("normalize", [True, False]),
        "feature_weighting": optuna_trial.suggest_categorical("feature_weighting", ["none", "TF-IDF", "BM25"]),
    }

    if params["feature_weighting"] == "BM25":
        params["BM25_k1"] = optuna_trial.suggest_float("BM25_k1", 0.5, 2.0)
        params["BM25_b"] = optuna_trial.suggest_float("BM25_b", 0.0, 1.0)
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = UserKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [9]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=30
)

[I 2025-11-20 16:27:59,426] Using an existing study with name 'UserKNNCFRecommender_tversky' instead of creating a new one.


  0%|          | 0/30 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 1734.16 column/sec. Elapsed time 15.62 sec
  Fold 1/5 - Score: 0.21540385484695435
Similarity column 27095 (100.0%), 1613.90 column/sec. Elapsed time 16.79 sec
  Fold 2/5 - Score: 0.21533000469207764
Similarity column 27095 (100.0%), 1213.34 column/sec. Elapsed time 22.33 sec
  Fold 3/5 - Score: 0.216003879904747
Similarity column 27095 (100.0%), 1496.23 column/sec. Elapsed time 18.11 sec
  Fold 4/5 - Score: 0.21528546512126923
[I 2025-11-20 16:29:57,978] Trial 76 finished with value: 0.21550580859184265 and parameters: {'topK': 407, 'shrink': 24, 'tversky_alpha': 0.23661189075145012, 'tversky_beta': 0.39850743105373065, 'normalize': True, 'feature_weighting': 'TF-IDF'}. Best is trial 12 with value: 0.2222549468278885.
Similarity column 27095 (100.0%), 1293.11 column/sec. Elapsed time 20.95 sec
  Fold 1/5 - Score: 0.19397315382957458
Similarity column 27095 (100.0%), 1396.85 column/sec. Elapsed time 19.40 sec
  Fold 2/5 - Score: 0.19474704563617706
Sim

In [10]:
optuna.visualization.plot_optimization_history(optuna_study)

In [11]:
optuna.visualization.plot_param_importances(optuna_study)

In [12]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

# **Hyperparameter tuning**

In [8]:
def refined_objective(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "similarity": "tversky",
        "topK": optuna_trial.suggest_int("topK", 320, 380),
        "shrink": optuna_trial.suggest_int("shrink", 0, 30),
        "tversky_alpha": optuna_trial.suggest_float("tversky_alpha", 0.22, 0.24),
        "tversky_beta": optuna_trial.suggest_float("tversky_beta", 0.8, 0.92),
        "normalize": False,
        "feature_weighting": "TF-IDF",
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        recommender_instance = UserKNNCFRecommender(URM_train)
        recommender_instance.fit(**params)
        
        # Evaluate
        score = evaluate_recommender(recommender_instance, at=20, URM_validation=URM_validation)
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [9]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME+"_refined",
    objective_function=refined_objective,
    n_trials=20
)

[I 2025-11-20 18:07:19,000] Using an existing study with name 'UserKNNCFRecommender_tversky_refined' instead of creating a new one.


  0%|          | 0/20 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 1800.49 column/sec. Elapsed time 15.05 sec
  Fold 1/5 - Score: 0.2201356291770935
Similarity column 27095 (100.0%), 1753.56 column/sec. Elapsed time 15.45 sec
  Fold 2/5 - Score: 0.22050254046916962
Similarity column 27095 (100.0%), 1684.34 column/sec. Elapsed time 16.09 sec
  Fold 3/5 - Score: 0.22240619361400604
Similarity column 27095 (100.0%), 1659.11 column/sec. Elapsed time 16.33 sec
  Fold 4/5 - Score: 0.21995872259140015
[I 2025-11-20 18:08:54,283] Trial 2 finished with value: 0.22075077891349792 and parameters: {'topK': 332, 'shrink': 30, 'tversky_alpha': 0.23842164849761838, 'tversky_beta': 0.8420613260124167}. Best is trial 0 with value: 0.22234492003917694.
Similarity column 27095 (100.0%), 1640.27 column/sec. Elapsed time 16.52 sec
  Fold 1/5 - Score: 0.22005560994148254
Similarity column 27095 (100.0%), 1673.77 column/sec. Elapsed time 16.19 sec
  Fold 2/5 - Score: 0.2201460748910904
Similarity column 27095 (100.0%), 1608.18 column/sec. E

In [10]:
optuna.visualization.plot_optimization_history(optuna_study)

In [11]:
optuna.visualization.plot_param_importances(optuna_study)

In [12]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

## **Best Model**
- Trial 1:
Best Value: 0.22270536422729492
Best Params: {'topK': 354, 'shrink': 17, 'tversky_alpha': 0.22981698409226547, 'tversky_beta': 0.8838363460495166, 'normalize': False, 'feature_weighting': 'TF-IDF'}

- Trial 2:
Best Value: 0.22263076901435852
Best Params: {'topK': 344, 'shrink': 14, 'tversky_alpha': 0.23851430007391616, 'tversky_beta': 0.8300200160081194}

Best Value: 0.22270536422729492

Best Params: {'topK': 354, 'shrink': 17, 'tversky_alpha': 0.22981698409226547, 'tversky_beta': 0.8838363460495166, 'normalize': False, 'feature_weighting': 'TF-IDF'}